# Module 5 • Neural Networks for Natural Language Processing

# Lesson 26 • Neural Network Foundations for NLP

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 140–180 minutes

---

## Scope

This lesson introduces the mathematical and computational foundations needed
for neural NLP models. It develops tensors, affine transformations,
activation functions, softmax classification, cross-entropy loss,
backpropagation, batching, optimization, embedding layers, regularization,
early stopping, and a complete neural text-classification model implemented
with NumPy.

No external dataset or model download is required.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain scalars, vectors, matrices, and higher-order tensors;
- track tensor shapes through a neural network;
- implement a dense layer;
- compare ReLU, tanh, and sigmoid activations;
- calculate softmax probabilities;
- calculate cross-entropy loss;
- explain forward propagation and backpropagation;
- derive gradients for a two-layer classifier;
- verify gradients numerically;
- create mini-batches;
- apply gradient descent;
- explain learning rate, initialization, and normalization;
- build and use an embedding lookup table;
- pool token embeddings into document vectors;
- train a neural text classifier from scratch;
- evaluate learning curves and classification errors;
- explain regularization, dropout, and early stopping;
- discuss Arabic tokenization and neural embedding considerations.

## Table of Contents

1. Why Neural Networks for NLP?
2. Scalars, Vectors, Matrices, and Tensors
3. Shape Reasoning
4. The Dense Layer
5. Activation Functions
6. Sigmoid
7. Tanh
8. ReLU
9. Softmax
10. Cross-Entropy Loss
11. Forward Propagation
12. Computational Graphs
13. Gradient Descent
14. Backpropagation
15. Gradient Checking
16. Parameter Initialization
17. Mini-Batches
18. Learning Rates
19. Training and Validation Splits
20. Embedding Layers
21. Padding and Masks
22. Pooling Token Embeddings
23. Neural Text-Classification Dataset
24. Vocabulary Construction
25. Encoding Documents
26. Two-Layer Neural Classifier
27. Training the Model
28. Evaluating the Model
29. Error Analysis
30. Overfitting
31. L2 Regularization
32. Dropout
33. Early Stopping
34. Class Imbalance
35. Numerical Stability
36. Arabic and Multilingual Considerations
37. Reproducibility and Reporting
38. Knowledge Check
39. Exercises
40. Summary and Next Lesson

# 1. Why Neural Networks for NLP?

Classical NLP systems often depend on manually designed features. Neural
networks learn layered transformations from data.

A simplified neural NLP pipeline is:

```text
tokens
  ↓
embedding lookup
  ↓
sequence or pooling layer
  ↓
hidden representation
  ↓
output layer
```

In [ ]:
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

neural_nlp_components = pd.DataFrame(
    [
        ("Embedding layer", "maps token IDs to dense vectors"),
        ("Hidden layer", "learns nonlinear feature combinations"),
        ("Output layer", "produces task predictions"),
        ("Loss function", "measures prediction error"),
        ("Optimizer", "updates parameters from gradients"),
    ],
    columns=["Component", "Purpose"],
)

neural_nlp_components

Neural networks do not remove the need for careful data design, evaluation,
and linguistic analysis.

# 2. Scalars, Vectors, Matrices, and Tensors

- **Scalar:** one number.
- **Vector:** one-dimensional array.
- **Matrix:** two-dimensional array.
- **Tensor:** general multidimensional array.

In [ ]:
scalar = np.array(3.5)
vector = np.array([1.0, 2.0, 3.0])
matrix = np.array(
    [
        [1.0, 2.0],
        [3.0, 4.0],
    ]
)
tensor_3d = np.zeros(
    (2, 3, 4)
)

shape_summary = pd.DataFrame(
    [
        ("scalar", scalar.shape, scalar.ndim),
        ("vector", vector.shape, vector.ndim),
        ("matrix", matrix.shape, matrix.ndim),
        ("3D tensor", tensor_3d.shape, tensor_3d.ndim),
    ],
    columns=["Object", "Shape", "Number of dimensions"],
)

shape_summary

In NLP, common tensor shapes include:

```text
token IDs:              (batch, sequence_length)
embeddings:             (batch, sequence_length, embedding_dim)
sentence representations:(batch, hidden_dim)
class logits:           (batch, number_of_classes)
```

# 3. Shape Reasoning

Shape errors are among the most common neural-network implementation errors.

Suppose:

```text
X: (batch_size, input_dim)
W: (input_dim, output_dim)
b: (output_dim,)
```

Then:

```text
X @ W + b → (batch_size, output_dim)
```

In [ ]:
batch_size = 4
input_dimension = 6
output_dimension = 3

X = np.random.default_rng(42).normal(
    size=(batch_size, input_dimension)
)
W = np.random.default_rng(43).normal(
    size=(input_dimension, output_dimension)
)
b = np.zeros(output_dimension)

output = X @ W + b

print("X shape:", X.shape)
print("W shape:", W.shape)
print("b shape:", b.shape)
print("Output shape:", output.shape)

Broadcasting allows the bias vector to be added to every row.

# 4. The Dense Layer

A dense or fully connected layer applies:

\[
Z = XW + b
\]

where \(X\) is the input, \(W\) contains weights, and \(b\) contains biases.

In [ ]:
def dense_forward(
    inputs: np.ndarray,
    weights: np.ndarray,
    biases: np.ndarray,
) -> np.ndarray:
    return inputs @ weights + biases


dense_output = dense_forward(
    X,
    W,
    b,
)

dense_output

A dense layer alone is linear. Stacking only linear layers is equivalent to one
larger linear transformation. Nonlinear activation functions are required to
learn nonlinear decision boundaries.

# 5. Activation Functions

Common activation functions include:

- sigmoid;
- tanh;
- ReLU;
- GELU;
- softmax for multiclass outputs.

In [ ]:
activation_summary = pd.DataFrame(
    [
        ("Sigmoid", "(0, 1)", "binary output, gates"),
        ("Tanh", "(-1, 1)", "recurrent hidden states"),
        ("ReLU", "[0, infinity)", "hidden layers"),
        ("Softmax", "probability simplex", "multiclass output"),
    ],
    columns=["Activation", "Range", "Typical role"],
)

activation_summary

# 6. Sigmoid

\[
sigmoid(x) = \frac{1}{1 + e^{-x}}
\]

In [ ]:
def sigmoid(values: np.ndarray) -> np.ndarray:
    clipped = np.clip(
        values,
        -30,
        30,
    )
    return 1.0 / (
        1.0 + np.exp(-clipped)
    )


x_values = np.linspace(
    -6,
    6,
    200,
)

plt.figure(figsize=(8, 5))
plt.plot(
    x_values,
    sigmoid(x_values),
)
plt.title("Sigmoid Activation")
plt.xlabel("Input")
plt.ylabel("Output")
plt.tight_layout()
plt.show()

Sigmoid saturates for large positive and negative values, which can produce very
small gradients.

# 7. Tanh

\[
tanh(x) =
\frac{e^x - e^{-x}}
{e^x + e^{-x}}
\]

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    x_values,
    np.tanh(x_values),
)
plt.title("Tanh Activation")
plt.xlabel("Input")
plt.ylabel("Output")
plt.tight_layout()
plt.show()

Tanh is zero-centered but can also saturate.

# 8. ReLU

\[
ReLU(x) = max(0, x)
\]

In [ ]:
def relu(values: np.ndarray) -> np.ndarray:
    return np.maximum(
        0.0,
        values,
    )


plt.figure(figsize=(8, 5))
plt.plot(
    x_values,
    relu(x_values),
)
plt.title("ReLU Activation")
plt.xlabel("Input")
plt.ylabel("Output")
plt.tight_layout()
plt.show()

ReLU is simple and effective, but neurons can become inactive if their inputs
remain negative.

# 9. Softmax

Softmax converts class logits into probabilities:

\[
softmax(z_i)
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
\]

In [ ]:
def softmax(
    logits: np.ndarray,
) -> np.ndarray:
    shifted = (
        logits
        - np.max(
            logits,
            axis=-1,
            keepdims=True,
        )
    )

    exponentials = np.exp(shifted)

    return (
        exponentials
        / np.sum(
            exponentials,
            axis=-1,
            keepdims=True,
        )
    )


example_logits = np.array(
    [
        [2.0, 1.0, 0.1],
        [0.5, 0.5, 0.5],
    ]
)

probabilities = softmax(
    example_logits
)

print(probabilities)
print(
    "Row sums:",
    probabilities.sum(axis=1),
)

Subtracting the largest logit prevents numerical overflow without changing the
probabilities.

# 10. Cross-Entropy Loss

For a gold class \(y\):

\[
L = -\log p(y)
\]

In [ ]:
def cross_entropy_loss(
    probabilities: np.ndarray,
    labels: np.ndarray,
) -> float:
    selected = probabilities[
        np.arange(len(labels)),
        labels,
    ]

    return float(
        -np.mean(
            np.log(
                np.clip(
                    selected,
                    1e-12,
                    1.0,
                )
            )
        )
    )


example_labels = np.array([0, 2])

print(
    "Cross-entropy:",
    cross_entropy_loss(
        probabilities,
        example_labels,
    ),
)

Confident correct predictions receive low loss. Confident incorrect predictions
receive high loss.

# 11. Forward Propagation

A two-layer network can be written as:

```text
Z1 = XW1 + b1
H  = ReLU(Z1)
Z2 = HW2 + b2
P  = softmax(Z2)
```

In [ ]:
generator = np.random.default_rng(42)

X_demo = generator.normal(
    size=(5, 8)
)

W1_demo = generator.normal(
    0.0,
    0.2,
    size=(8, 6),
)
b1_demo = np.zeros(6)

W2_demo = generator.normal(
    0.0,
    0.2,
    size=(6, 4),
)
b2_demo = np.zeros(4)

Z1_demo = X_demo @ W1_demo + b1_demo
H_demo = relu(Z1_demo)
Z2_demo = H_demo @ W2_demo + b2_demo
P_demo = softmax(Z2_demo)

print("Hidden shape:", H_demo.shape)
print("Probability shape:", P_demo.shape)

# 12. Computational Graphs

A computational graph records operations and dependencies.

```text
X ─┐
   ├─ matrix multiply ─ add bias ─ ReLU ─ output layer ─ softmax ─ loss
W ─┘
```

Backpropagation applies the chain rule through this graph in reverse order.

# 13. Gradient Descent

Gradient descent updates parameters:

\[
\theta_{new}
=
\theta_{old}
-
learning\_rate
\times
\nabla_\theta L
\]

In [ ]:
parameter = 4.0
learning_rate = 0.1

history = []

for step in range(20):
    loss = parameter ** 2
    gradient = 2 * parameter

    history.append(
        {
            "step": step,
            "parameter": parameter,
            "loss": loss,
        }
    )

    parameter -= (
        learning_rate
        * gradient
    )

pd.DataFrame(history).head()

In [ ]:
gradient_frame = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(
    gradient_frame["step"],
    gradient_frame["loss"],
)
plt.title("Gradient Descent on a Quadratic Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

# 14. Backpropagation

For softmax plus cross-entropy, the gradient with respect to logits is:

\[
\frac{\partial L}{\partial Z}
=
P - Y
\]

divided by batch size when using the mean loss.

In [ ]:
def one_hot(
    labels: np.ndarray,
    class_count: int,
) -> np.ndarray:
    encoded = np.zeros(
        (len(labels), class_count)
    )
    encoded[
        np.arange(len(labels)),
        labels,
    ] = 1.0
    return encoded


demo_labels = np.array(
    [0, 1, 3, 2, 1]
)

gold_one_hot = one_hot(
    demo_labels,
    class_count=4,
)

gradient_logits = (
    P_demo
    - gold_one_hot
) / len(demo_labels)

print(gradient_logits.shape)

ReLU derivative:

\[
ReLU'(x)
=
\begin{cases}
1 & x > 0 \\
0 & x \leq 0
\end{cases}
\]

In [ ]:
def relu_derivative(
    values: np.ndarray,
) -> np.ndarray:
    return (
        values > 0
    ).astype(float)


dW2_demo = H_demo.T @ gradient_logits
db2_demo = gradient_logits.sum(axis=0)

dH_demo = gradient_logits @ W2_demo.T
dZ1_demo = (
    dH_demo
    * relu_derivative(
        Z1_demo
    )
)

dW1_demo = X_demo.T @ dZ1_demo
db1_demo = dZ1_demo.sum(axis=0)

print("dW1:", dW1_demo.shape)
print("db1:", db1_demo.shape)
print("dW2:", dW2_demo.shape)
print("db2:", db2_demo.shape)

# 15. Gradient Checking

Numerical gradient checking compares an analytical gradient with a finite
difference approximation.

In [ ]:
def numerical_gradient(
    function,
    value: float,
    epsilon: float = 1e-5,
) -> float:
    return (
        function(value + epsilon)
        - function(value - epsilon)
    ) / (
        2 * epsilon
    )


function = lambda x: x ** 3 + 2 * x
point = 1.7

numerical = numerical_gradient(
    function,
    point,
)

analytical = (
    3 * point ** 2
    + 2
)

print("Numerical:", numerical)
print("Analytical:", analytical)
print(
    "Difference:",
    abs(numerical - analytical),
)

Gradient checking is useful for debugging small models but too expensive for
routine large-scale training.

# 16. Parameter Initialization

Poor initialization can cause unstable activations or gradients.

Common strategies include:

- small random values;
- Xavier/Glorot initialization;
- He initialization;
- pretrained embeddings.

In [ ]:
def he_initialization(
    input_dim: int,
    output_dim: int,
    generator: np.random.Generator,
) -> np.ndarray:
    scale = math.sqrt(
        2.0 / input_dim
    )

    return generator.normal(
        0.0,
        scale,
        size=(
            input_dim,
            output_dim,
        ),
    )


initialized = he_initialization(
    100,
    64,
    np.random.default_rng(42),
)

print(
    "Weight standard deviation:",
    initialized.std(),
)

He initialization is commonly paired with ReLU activations.

# 17. Mini-Batches

Mini-batch training processes a subset of examples per update.

In [ ]:
def batch_indices(
    example_count: int,
    batch_size: int,
    generator: np.random.Generator,
    shuffle: bool = True,
):
    indices = np.arange(
        example_count
    )

    if shuffle:
        generator.shuffle(indices)

    for start in range(
        0,
        example_count,
        batch_size,
    ):
        yield indices[
            start:start + batch_size
        ]


batches = list(
    batch_indices(
        example_count=10,
        batch_size=4,
        generator=np.random.default_rng(42),
    )
)

batches

Mini-batches balance computational efficiency, gradient noise, and memory use.

# 18. Learning Rates

A learning rate that is too small produces slow training. A rate that is too
large can make the loss oscillate or diverge.

In [ ]:
learning_rate_effects = pd.DataFrame(
    [
        ("Too small", "slow convergence"),
        ("Appropriate", "stable progress"),
        ("Too large", "oscillation or divergence"),
        ("Decayed", "large early steps, smaller later steps"),
    ],
    columns=["Learning-rate behavior", "Likely effect"],
)

learning_rate_effects

# 19. Training and Validation Splits

Neural models require distinct partitions:

- training set for parameter updates;
- validation set for model selection and early stopping;
- test set for final evaluation.

Test data must not influence hyperparameters, stopping time, vocabulary
thresholds, or model selection.

# 20. Embedding Layers

An embedding layer is a trainable lookup table.

If the vocabulary size is \(V\) and embedding dimension is \(D\):

```text
embedding table shape = (V, D)
```

In [ ]:
vocabulary_size = 12
embedding_dimension = 5

embedding_table = np.random.default_rng(
    42
).normal(
    0.0,
    0.1,
    size=(
        vocabulary_size,
        embedding_dimension,
    ),
)

token_ids = np.array(
    [
        [2, 5, 7],
        [4, 3, 1],
    ]
)

embedded_tokens = embedding_table[
    token_ids
]

print(
    "Token ID shape:",
    token_ids.shape,
)
print(
    "Embedding shape:",
    embedded_tokens.shape,
)

The lookup operation selects rows; it does not multiply by a one-hot matrix in
practice.

# 21. Padding and Masks

Sentences in one batch often have different lengths. Padding adds a special
token to shorter sequences.

In [ ]:
PAD_ID = 0

padded_sequences = np.array(
    [
        [2, 4, 7, 9],
        [3, 8, 0, 0],
        [6, 5, 1, 0],
    ]
)

padding_mask = (
    padded_sequences
    != PAD_ID
)

print(padded_sequences)
print()
print(padding_mask.astype(int))

Masks prevent padding tokens from contributing to pooling, attention, or loss.

# 22. Pooling Token Embeddings

Masked mean pooling:

\[
document =
\frac{
\sum_t mask_t \times embedding_t
}{
\sum_t mask_t
}
\]

In [ ]:
def masked_mean_pool(
    embedded: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    expanded_mask = mask[..., None]

    summed = np.sum(
        embedded
        * expanded_mask,
        axis=1,
    )

    counts = np.sum(
        expanded_mask,
        axis=1,
    )

    return (
        summed
        / np.maximum(
            counts,
            1,
        )
    )


padded_embeddings = embedding_table[
    padded_sequences
]

pooled_documents = masked_mean_pool(
    padded_embeddings,
    padding_mask,
)

print(pooled_documents.shape)

# 23. Neural Text-Classification Dataset

We use four balanced classes:

- health;
- finance;
- technology;
- travel.

In [ ]:
records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medical care", "health"),
    ("exercise improves patient health", "health"),
    ("clinic schedules treatment", "health"),
    ("medicine supports recovery", "health"),
    ("hospital reviews diagnosis", "health"),
    ("patient requests medical help", "health"),
    ("nutrition improves health", "health"),
    ("doctor and nurse work together", "health"),
    ("treatment begins today", "health"),
    ("medical service needs information", "health"),
    ("clinic helps the patient", "health"),

    ("bank approves the loan", "finance"),
    ("invoice contains a charge", "finance"),
    ("card payment failed", "finance"),
    ("customer requests a refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased", "finance"),
    ("bank transfers money", "finance"),
    ("payment needs approval", "finance"),
    ("refund request is pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("card account needs help", "finance"),
    ("billing service changed today", "finance"),

    ("software update caused error", "technology"),
    ("application cannot connect to server", "technology"),
    ("network upload failed", "technology"),
    ("computer needs system update", "technology"),
    ("device cannot install software", "technology"),
    ("server lost data", "technology"),
    ("application displays an error", "technology"),
    ("network service is unavailable", "technology"),
    ("computer system needs help", "technology"),
    ("upload problem started today", "technology"),
    ("software request failed", "technology"),
    ("device update is pending", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel", "travel"),
    ("luggage is missing", "travel"),
    ("travel ticket changed", "travel"),
    ("airport delays the flight", "travel"),
    ("hotel reservation needs help", "travel"),
    ("tourist visits museum", "travel"),
    ("journey reaches the city", "travel"),
    ("beach trip starts today", "travel"),
    ("flight service changed", "travel"),
    ("ticket request is pending", "travel"),
    ("airport lost the luggage", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 24. Vocabulary Construction

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(
    text: str,
) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


X_train_full, X_test, y_train_full, y_test = (
    train_test_split(
        dataset["text"],
        dataset["label"],
        test_size=0.25,
        random_state=42,
        stratify=dataset["label"],
    )
)

X_train, X_validation, y_train, y_validation = (
    train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.25,
        random_state=42,
        stratify=y_train_full,
    )
)

training_counts = Counter(
    token
    for text in X_train
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
] + sorted(training_counts)

word_to_index = {
    word: index
    for index, word in enumerate(
        vocabulary
    )
}

PAD_ID = word_to_index["<PAD>"]
UNK_ID = word_to_index["<UNK>"]

print("Vocabulary size:", len(vocabulary))
print("Train:", len(X_train))
print("Validation:", len(X_validation))
print("Test:", len(X_test))

The vocabulary is built only from the training split.

# 25. Encoding Documents

In [ ]:
MAX_LENGTH = 8


def encode_text(
    text: str,
    max_length: int = MAX_LENGTH,
) -> tuple[np.ndarray, np.ndarray]:
    token_ids = [
        word_to_index.get(
            token,
            UNK_ID,
        )
        for token in tokenize(text)
    ][:max_length]

    mask = [1] * len(token_ids)

    while len(token_ids) < max_length:
        token_ids.append(PAD_ID)
        mask.append(0)

    return (
        np.asarray(
            token_ids,
            dtype=int,
        ),
        np.asarray(
            mask,
            dtype=float,
        ),
    )


def encode_collection(
    texts,
) -> tuple[np.ndarray, np.ndarray]:
    encoded = [
        encode_text(text)
        for text in texts
    ]

    ids = np.vstack(
        [
            item[0]
            for item in encoded
        ]
    )

    masks = np.vstack(
        [
            item[1]
            for item in encoded
        ]
    )

    return ids, masks


train_ids, train_masks = encode_collection(
    X_train
)
validation_ids, validation_masks = encode_collection(
    X_validation
)
test_ids, test_masks = encode_collection(
    X_test
)

label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(
    y_train
)
validation_labels = label_encoder.transform(
    y_validation
)
test_labels = label_encoder.transform(
    y_test
)

print(train_ids.shape)
print(train_masks.shape)

# 26. Two-Layer Neural Classifier

Architecture:

```text
token IDs
  ↓
embedding lookup
  ↓
masked mean pooling
  ↓
dense hidden layer + ReLU
  ↓
dense output layer
  ↓
softmax
```

In [ ]:
def initialize_model(
    vocabulary_size: int,
    embedding_dim: int,
    hidden_dim: int,
    class_count: int,
    seed: int = 42,
) -> dict[str, np.ndarray]:
    generator = np.random.default_rng(
        seed
    )

    embeddings = generator.normal(
        0.0,
        0.1,
        size=(
            vocabulary_size,
            embedding_dim,
        ),
    )

    embeddings[PAD_ID] = 0.0

    return {
        "embeddings": embeddings,
        "W1": he_initialization(
            embedding_dim,
            hidden_dim,
            generator,
        ),
        "b1": np.zeros(hidden_dim),
        "W2": generator.normal(
            0.0,
            math.sqrt(
                1.0 / hidden_dim
            ),
            size=(
                hidden_dim,
                class_count,
            ),
        ),
        "b2": np.zeros(class_count),
    }


model = initialize_model(
    vocabulary_size=len(vocabulary),
    embedding_dim=16,
    hidden_dim=12,
    class_count=len(
        label_encoder.classes_
    ),
)

{
    key: value.shape
    for key, value in model.items()
}

In [ ]:
def forward_model(
    token_ids: np.ndarray,
    masks: np.ndarray,
    parameters: dict[str, np.ndarray],
) -> tuple[
    np.ndarray,
    dict[str, np.ndarray],
]:
    embedded = parameters[
        "embeddings"
    ][token_ids]

    pooled = masked_mean_pool(
        embedded,
        masks,
    )

    Z1 = (
        pooled
        @ parameters["W1"]
        + parameters["b1"]
    )

    hidden = relu(Z1)

    logits = (
        hidden
        @ parameters["W2"]
        + parameters["b2"]
    )

    probabilities = softmax(logits)

    cache = {
        "token_ids": token_ids,
        "masks": masks,
        "embedded": embedded,
        "pooled": pooled,
        "Z1": Z1,
        "hidden": hidden,
        "probabilities": probabilities,
    }

    return probabilities, cache

In [ ]:
def backward_model(
    labels: np.ndarray,
    cache: dict[str, np.ndarray],
    parameters: dict[str, np.ndarray],
    l2_strength: float = 0.0,
) -> dict[str, np.ndarray]:
    batch_size = len(labels)

    targets = one_hot(
        labels,
        class_count=parameters[
            "b2"
        ].shape[0],
    )

    d_logits = (
        cache["probabilities"]
        - targets
    ) / batch_size

    dW2 = (
        cache["hidden"].T
        @ d_logits
        + l2_strength
        * parameters["W2"]
    )
    db2 = d_logits.sum(axis=0)

    d_hidden = (
        d_logits
        @ parameters["W2"].T
    )

    dZ1 = (
        d_hidden
        * relu_derivative(
            cache["Z1"]
        )
    )

    dW1 = (
        cache["pooled"].T
        @ dZ1
        + l2_strength
        * parameters["W1"]
    )
    db1 = dZ1.sum(axis=0)

    d_pooled = (
        dZ1
        @ parameters["W1"].T
    )

    counts = np.maximum(
        cache["masks"].sum(
            axis=1,
            keepdims=True,
        ),
        1.0,
    )

    token_gradient = (
        d_pooled[:, None, :]
        / counts[:, :, None]
    ) * (
        cache["masks"][:, :, None]
    )

    d_embeddings = np.zeros_like(
        parameters["embeddings"]
    )

    for row in range(
        cache["token_ids"].shape[0]
    ):
        for position in range(
            cache["token_ids"].shape[1]
        ):
            token_id = cache[
                "token_ids"
            ][row, position]

            if token_id == PAD_ID:
                continue

            d_embeddings[token_id] += (
                token_gradient[
                    row,
                    position,
                ]
            )

    d_embeddings[PAD_ID] = 0.0

    return {
        "embeddings": d_embeddings,
        "W1": dW1,
        "b1": db1,
        "W2": dW2,
        "b2": db2,
    }

# 27. Training the Model

In [ ]:
def evaluate_model(
    token_ids: np.ndarray,
    masks: np.ndarray,
    labels: np.ndarray,
    parameters: dict[str, np.ndarray],
) -> tuple[
    float,
    float,
    np.ndarray,
    np.ndarray,
]:
    probabilities, _ = forward_model(
        token_ids,
        masks,
        parameters,
    )

    loss = cross_entropy_loss(
        probabilities,
        labels,
    )

    predictions = probabilities.argmax(
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions,
    )

    return (
        loss,
        accuracy,
        predictions,
        probabilities,
    )


def train_model(
    train_ids: np.ndarray,
    train_masks: np.ndarray,
    train_labels: np.ndarray,
    validation_ids: np.ndarray,
    validation_masks: np.ndarray,
    validation_labels: np.ndarray,
    parameters: dict[str, np.ndarray],
    epochs: int = 160,
    batch_size: int = 8,
    learning_rate: float = 0.05,
    l2_strength: float = 1e-4,
    patience: int = 20,
    seed: int = 42,
):
    generator = np.random.default_rng(
        seed
    )

    history = []
    best_parameters = {
        key: value.copy()
        for key, value in parameters.items()
    }
    best_validation_loss = float(
        "inf"
    )
    epochs_without_improvement = 0

    for epoch in range(epochs):
        current_rate = (
            learning_rate
            * (
                1.0
                - 0.75
                * epoch
                / max(
                    epochs - 1,
                    1,
                )
            )
        )

        for indices in batch_indices(
            len(train_ids),
            batch_size,
            generator,
            shuffle=True,
        ):
            probabilities, cache = (
                forward_model(
                    train_ids[indices],
                    train_masks[indices],
                    parameters,
                )
            )

            gradients = backward_model(
                train_labels[indices],
                cache,
                parameters,
                l2_strength=l2_strength,
            )

            for key in parameters:
                parameters[key] -= (
                    current_rate
                    * gradients[key]
                )

            parameters[
                "embeddings"
            ][PAD_ID] = 0.0

        (
            train_loss,
            train_accuracy,
            _,
            _,
        ) = evaluate_model(
            train_ids,
            train_masks,
            train_labels,
            parameters,
        )

        (
            validation_loss,
            validation_accuracy,
            _,
            _,
        ) = evaluate_model(
            validation_ids,
            validation_masks,
            validation_labels,
            parameters,
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
                "train_accuracy": train_accuracy,
                "validation_accuracy": validation_accuracy,
            }
        )

        if (
            validation_loss
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_loss
            )

            best_parameters = {
                key: value.copy()
                for key, value in parameters.items()
            }

            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            break

    return (
        best_parameters,
        pd.DataFrame(history),
    )


trained_model, training_history = train_model(
    train_ids,
    train_masks,
    train_labels,
    validation_ids,
    validation_masks,
    validation_labels,
    model,
)

print(
    "Epochs completed:",
    len(training_history),
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["train_loss"],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Neural Text Classifier Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["train_accuracy"],
    label="Training accuracy",
)
plt.plot(
    training_history["epoch"],
    training_history["validation_accuracy"],
    label="Validation accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# 28. Evaluating the Model

In [ ]:
(
    test_loss,
    test_accuracy,
    test_predictions,
    test_probabilities,
) = evaluate_model(
    test_ids,
    test_masks,
    test_labels,
    trained_model,
)

test_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
)

print(
    "Test loss:",
    round(test_loss, 3),
)
print(
    "Test accuracy:",
    round(test_accuracy, 3),
)
print(
    "Test macro F1:",
    round(test_macro_f1, 3),
)

In [ ]:
predicted_labels = (
    label_encoder.inverse_transform(
        test_predictions
    )
)

print(
    classification_report(
        y_test,
        predicted_labels,
        zero_division=0,
    )
)

In [ ]:
class_names = list(
    label_encoder.classes_
)

confusion = confusion_matrix(
    y_test,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    confusion,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 29. Error Analysis

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": X_test.reset_index(
            drop=True
        ),
        "actual": y_test.reset_index(
            drop=True
        ),
        "predicted": predicted_labels,
        "confidence": test_probabilities.max(
            axis=1
        ),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

Review errors for:

- OOV words;
- ambiguous vocabulary;
- short texts;
- mixed domains;
- insufficient training examples;
- label noise;
- truncation caused by maximum sequence length.

# 30. Overfitting

Overfitting occurs when training performance improves while validation
performance degrades.

In [ ]:
overfitting_signals = pd.DataFrame(
    [
        ("Training loss decreases", "expected"),
        ("Validation loss increases", "possible overfitting"),
        ("Training accuracy much higher", "capacity exceeds generalization"),
        ("Unstable validation metrics", "small or noisy validation set"),
    ],
    columns=["Observation", "Interpretation"],
)

overfitting_signals

# 31. L2 Regularization

L2 regularization adds a penalty:

\[
L_{total}
=
L_{task}
+
\lambda ||W||_2^2
\]

L2 discourages large weights but does not guarantee better generalization.

# 32. Dropout

Dropout randomly masks hidden units during training.

Simplified inverted dropout:

```text
mask ~ Bernoulli(keep_probability)
output = hidden * mask / keep_probability
```

In [ ]:
def dropout(
    hidden: np.ndarray,
    dropout_rate: float,
    generator: np.random.Generator,
    training: bool = True,
) -> np.ndarray:
    if (
        not training
        or dropout_rate <= 0
    ):
        return hidden

    keep_probability = (
        1.0 - dropout_rate
    )

    mask = (
        generator.random(
            hidden.shape
        )
        < keep_probability
    )

    return (
        hidden
        * mask
        / keep_probability
    )


hidden_example = np.ones(
    (3, 8)
)

dropout(
    hidden_example,
    dropout_rate=0.5,
    generator=np.random.default_rng(42),
)

Dropout is disabled during evaluation.

# 33. Early Stopping

Early stopping keeps the model from the epoch with the best validation metric.

The training function above uses validation loss and a patience parameter.

Early stopping is model selection. The test set must remain untouched until the
final model is selected.

# 34. Class Imbalance

Imbalanced classes can cause a model to favor majority labels.

Possible responses:

- class weights;
- balanced sampling;
- macro-averaged metrics;
- more minority-class data;
- threshold adjustment.

In [ ]:
class_imbalance_responses = pd.DataFrame(
    [
        ("Class weights", "increase minority loss contribution"),
        ("Balanced batches", "change sampling frequency"),
        ("Macro F1", "weight classes equally in evaluation"),
        ("More data", "improve minority representation"),
    ],
    columns=["Method", "Purpose"],
)

class_imbalance_responses

# 35. Numerical Stability

Neural implementations must guard against:

- exponential overflow;
- logarithm of zero;
- division by zero;
- exploding gradients;
- NaN parameters.

In [ ]:
stability_checks = pd.Series(
    {
        "all_embeddings_finite": bool(
            np.isfinite(
                trained_model[
                    "embeddings"
                ]
            ).all()
        ),
        "all_W1_finite": bool(
            np.isfinite(
                trained_model["W1"]
            ).all()
        ),
        "all_W2_finite": bool(
            np.isfinite(
                trained_model["W2"]
            ).all()
        ),
        "padding_vector_zero": bool(
            np.allclose(
                trained_model[
                    "embeddings"
                ][PAD_ID],
                0.0,
            )
        ),
    },
    name="Numerical checks",
)

stability_checks

Gradient clipping is another common protection for sequence models.

# 36. Arabic and Multilingual Considerations

Neural Arabic NLP must account for:

- attached clitics;
- rich morphology;
- diacritics;
- orthographic variation;
- MSA and dialects;
- Arabizi and code-switching;
- segmentation choices;
- vocabulary fragmentation.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        ("وَبِالْمَدْرَسَةِ", "وَ + بِ + الْمَدْرَسَةِ"),
        ("وَالطَّبِيبُ", "وَ + الطَّبِيبُ"),
        ("كِتَابُهُ", "كِتَابُ + هُ"),
    ],
    columns=[
        "Surface form",
        "Illustrative segmentation",
    ],
)

arabic_examples

A word-level embedding table may treat every surface form as unrelated. Subword
and character-aware models can share information, but the tokenization policy
must remain explicit.

In [ ]:
arabic_sentence = (
    "الطبيب يعالج المريض في المستشفى"
)

arabic_tokens = arabic_sentence.split()

print("Whitespace tokens:", arabic_tokens)
print("Token count:", len(arabic_tokens))

For fully vocalized Arabic tasks, diacritics may carry morphological and lexical
information and should not be removed automatically.

# 37. Reproducibility and Reporting

Report:

- dataset and split;
- tokenization;
- vocabulary threshold;
- maximum sequence length;
- embedding dimension;
- hidden dimension;
- activation;
- initialization;
- optimizer;
- learning-rate schedule;
- batch size;
- regularization;
- early-stopping rule;
- random seed;
- evaluation metrics.

In [ ]:
import platform

metadata = pd.Series(
    {
        "dataset_examples": len(dataset),
        "classes": dataset["label"].nunique(),
        "vocabulary_size": len(vocabulary),
        "maximum_length": MAX_LENGTH,
        "embedding_dimension": trained_model[
            "embeddings"
        ].shape[1],
        "hidden_dimension": trained_model[
            "W1"
        ].shape[1],
        "activation": "ReLU",
        "optimizer": "mini-batch gradient descent",
        "early_stopping": True,
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
    },
    name="Neural NLP experiment",
)

metadata

# 38. Knowledge Check

1. What is a tensor?
2. Why is shape reasoning important?
3. What does a dense layer compute?
4. Why are nonlinear activations needed?
5. How do sigmoid, tanh, and ReLU differ?
6. What does softmax produce?
7. What does cross-entropy measure?
8. What is forward propagation?
9. What is backpropagation?
10. Why use gradient checking?
11. Why does initialization matter?
12. What is a mini-batch?
13. What does an embedding layer store?
14. Why are padding masks required?
15. What is early stopping?
16. How does L2 regularization affect weights?
17. How does dropout operate?
18. Why can training accuracy be misleading?
19. What numerical failures should be monitored?
20. Which Arabic properties affect neural NLP models?

# 39. Exercises

## Exercise 1 — Tensor Shapes

Trace shapes through several dense-layer architectures.

## Exercise 2 — Activations

Implement Leaky ReLU and GELU.

## Exercise 3 — Softmax

Compare stable and unstable softmax implementations.

## Exercise 4 — Backpropagation

Derive gradients for a one-hidden-layer network.

## Exercise 5 — Gradient Check

Numerically verify selected weight gradients.

## Exercise 6 — Initialization

Compare small random, Xavier, and He initialization.

## Exercise 7 — Batch Size

Compare full-batch, mini-batch, and single-example updates.

## Exercise 8 — Neural Text Classification

Add more examples and compare hidden dimensions.

## Exercise 9 — Dropout

Integrate dropout into the training function.

## Exercise 10 — Arabic Classification

Build a small Arabic classifier with explicit tokenization and diacritic
policy.

## Challenge Exercises

1. Implement Adam optimization.
2. Add gradient clipping.
3. Add class-weighted cross-entropy.
4. Initialize embeddings from Lesson 24 vectors.
5. Refactor the model into reusable Python classes.

# 40. Summary and Next Lesson

In this lesson:

- scalars, vectors, matrices, and tensors were distinguished;
- tensor shapes were traced through dense layers;
- sigmoid, tanh, ReLU, and softmax were implemented;
- cross-entropy measured multiclass prediction error;
- forward propagation produced neural predictions;
- backpropagation applied the chain rule;
- numerical gradient checking supported debugging;
- initialization, mini-batches, and learning rates shaped optimization;
- embedding tables mapped token IDs to dense vectors;
- padding and masks supported variable-length text;
- masked pooling created document representations;
- a complete NumPy neural text classifier was trained and evaluated;
- L2 regularization, dropout, and early stopping were introduced;
- numerical stability and class imbalance were addressed;
- Arabic tokenization and morphology were connected to neural model design.

## Next Lesson

**Lesson 27: Recurrent Neural Networks for Sequence Modeling** introduces
hidden states, sequence recurrence, unrolling through time, vanishing and
exploding gradients, and recurrent text classification.

# References

- Goodfellow, I., Bengio, Y., & Courville, A. *Deep Learning*.
- Goldberg, Y. *Neural Network Methods for Natural Language Processing*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- neural optimization, embedding-layer, and text-classification literature.